# Pose Classification using MediaPipe Landmark Detection

In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import json

In [ ]:
data = pd.read_csv('posedata_20260305_155724.csv')

X = data.drop('label', axis=1).values
y = data['label'].values

scaler = StandardScaler()
encoder = LabelEncoder()
y = encoder.fit_transform(y)

X = X.reshape(-1, 33, 2)

left_hip = X[:, 24]
right_hip = X[:, 23]

hip_center = (left_hip + right_hip) / 2
X = X - hip_center[:, None, :]

left_shoulder = X[:, 12]
right_shoulder = X[:, 11]

shoulder_center = (left_shoulder + right_shoulder) / 2

torso_length = np.linalg.norm(shoulder_center, axis=1)
torso_length = torso_length[:, None, None]

X = X / torso_length

X = X.reshape(-1, 66)

X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train).float()
y_train_tensor = torch.tensor(y_train).long()

X_test_tensor = torch.tensor(X_test).float()
y_test_tensor = torch.tensor(y_test).long()

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32)

class PoseClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(66, 128),
            #nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        return self.model(x)

model = PoseClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
learning_rate = 1e-3
batch_size = 64
epochs = 5

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, criterion, optimizer)
    test_loop(test_loader, model, criterion)
print("Done!")



Epoch 1
-------------------------------
loss: 1.376865  [   32/18453]
loss: 0.019729  [ 6432/18453]
loss: 0.012884  [12832/18453]
loss: 0.000610  [19232/18453]
loss: 0.000502  [25632/18453]
loss: 0.033930  [32032/18453]
Test Error: 
 Accuracy: 100.0%, Avg loss: 0.000616 

Epoch 2
-------------------------------
loss: 0.000806  [   32/18453]
loss: 0.000209  [ 6432/18453]
loss: 0.000256  [12832/18453]
loss: 0.000350  [19232/18453]
loss: 0.000064  [25632/18453]
loss: 0.000333  [32032/18453]
Test Error: 
 Accuracy: 100.0%, Avg loss: 0.000060 

Epoch 3
-------------------------------
loss: 0.000313  [   32/18453]
loss: 0.000219  [ 6432/18453]
loss: 0.000061  [12832/18453]
loss: 0.000025  [19232/18453]
loss: 0.000019  [25632/18453]
loss: 0.000058  [32032/18453]
Test Error: 
 Accuracy: 100.0%, Avg loss: 0.000019 

Epoch 4
-------------------------------
loss: 0.000029  [   32/18453]
loss: 0.000083  [ 6432/18453]
loss: 0.000012  [12832/18453]
loss: 0.000217  [19232/18453]
loss: 0.000021  [2563

In [ ]:
class PoseClassifierWithSoftmax(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        logits = self.base_model(x)
        return self.softmax(logits)

model.eval()

export_model = PoseClassifierWithSoftmax(model)
export_model.eval()

torch.onnx.export(
    export_model,
    torch.randn(1, 66),
    "pose_classifier.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)

# scaler and labels export remain the same
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "std": scaler.scale_.tolist()
}
with open("pose_scaler.json", "w") as f:
    json.dump(scaler_params, f)

label_mapping = {}
for i, name in enumerate(encoder.classes_):
    label_mapping[str(i)] = name

with open("pose_labels.json", "w") as f:
    json.dump(label_mapping, f)

print("Exported with softmax baked in — output is now probabilities [0, 1]")
print(f"Labels: {label_mapping}")

Exported with softmax baked in — output is now probabilities [0, 1]
Labels: {'0': 'air', '1': 'earth', '2': 'fire', '3': 'water'}
